# Notebook 35 -- Open-loop vs. Plant-wide Control Comparison

Ablation study: how does the control structure affect posterior identifiability?
1. Closed-loop (5 PI loops) vs. open-loop for L1, L2, L10.
2. Compare posterior width for each parameter.
3. Confirm that plant-wide control reduces identifiability for beta_r, beta_s.


In [ ]:
import sys; sys.path.insert(0, '../src')
import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pickle

from cstr_sbi.luyben.priors import PARAM_NAMES
from cstr_sbi.luyben.inference import sample_posterior
from cstr_sbi.luyben.summaries import compute_summary_statistics
from cstr_sbi.luyben.simulator import simulate_em_window, warm_start_ic, apply_sensor_layer
from cstr_sbi.luyben.physics import NOMINAL_INLET, NOMINAL_CTRL_ALL, NOMINAL_THETA
from cstr_sbi.luyben.scenarios import SCENARIO_CONFIGS

with open('../results/luyben_posterior.pkl', 'rb') as f:
    posterior = pickle.load(f)['posterior']


## Closed-loop vs open-loop posterior widths

In [ ]:
# For each of L1, L2, L10: generate one closed-loop and one open-loop window,
# compute summaries, sample posterior, compare 90% CI widths.
scenarios_to_compare = [1, 2, 10]
param_names_list = list(PARAM_NAMES)

for sc_id in scenarios_to_compare:
    sc = next(s for s in SCENARIO_CONFIGS.values() if s.id == sc_id)
    theta = sc.theta()
    y0 = warm_start_ic(theta)
    proc_key, sens_key = jax.random.split(jax.random.PRNGKey(sc_id * 100))

    # Closed-loop window
    _, _, obs_cl = simulate_em_window(theta, NOMINAL_INLET, NOMINAL_CTRL_ALL, y0, key=proc_key)
    obs_cl_noisy = apply_sensor_layer(obs_cl, key=sens_key)
    t_out = jnp.arange(1, obs_cl.shape[0]+1) * 1.0
    s_cl = np.asarray(compute_summary_statistics(obs_cl_noisy, t_out))
    samples_cl = sample_posterior(posterior, s_cl, n_samples=5000)

    print(f'\nL{sc_id} ({sc.name}):')
    print(f'  {'param':12s} {'true':>7s} {'CL mean':>8s} {'CL 90%CI':>10s}')
    for j, pname in enumerate(param_names_list):
        true_val = float(theta[j])
        cl_mean = np.mean(samples_cl[:, j])
        cl_lo, cl_hi = np.percentile(samples_cl[:, j], [5, 95])
        print(f'  {pname:12s} {true_val:7.3f} {cl_mean:8.3f} [{cl_lo:.3f}, {cl_hi:.3f}]')
